In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def achar_raiz():
    base = Path("/content/drive/MyDrive")
    candidatos = [base / "GS2-GENIA", *base.glob("*/GS2-GENIA"), *base.glob("*/*/GS2-GENIA")]
    for c in candidatos:
        if (c / "src" / "config.py").exists():
            return c
    return None

ROOT = achar_raiz()
assert ROOT, "Não achei a pasta GS2-GENIA no Drive (procurei até 2 níveis dentro de MyDrive)."
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

# o torchao do Colab é incompatível com o peft e impede carregar o adaptador
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44", "accelerate>=0.33",
                "peft>=0.12", "sentence-transformers>=3.0", "numpy"], check=True)
print("Dependências prontas.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projeto: /content/drive/MyDrive/GS2-GENIA
Dependências prontas.


In [8]:
import json, inspect
from pathlib import Path
from src import config
from src.rag import Generator, answer
from src.retrieval import Retriever, load_threshold

try:
    from src.llm import mascarar_pii          # mascara CPF, telefone, e-mail e CEP
except ImportError:
    mascarar_pii = lambda t: t

# trechos da base
chunks_path = config.CHUNKS_PATH if config.CHUNKS_PATH.exists() else config.ROOT / "base" / "chunks.jsonl"
chunks = [json.loads(l) for l in open(chunks_path, encoding="utf-8") if l.strip()]

# busca (guarda os embeddings no Drive para não recalcular a cada sessão, se a versão do projeto suportar)
kw = {}
if "cache_path" in inspect.signature(Retriever.__init__).parameters:
    kw["cache_path"] = Path("/content/drive/MyDrive/.cache_defesacivil/embeddings.npz")
retriever = Retriever(chunks, **kw)

# modelo com o adaptador LoRA treinado
assert (config.ADAPTER_DIR / "adapter_config.json").exists(), f"Adaptador não encontrado em {config.ADAPTER_DIR}"
gen = Generator(adapter_path=config.ADAPTER_DIR)
TH = load_threshold()
print(f"Pronto: {len(chunks)} trechos | limiar {TH:.3f} | LoRA ativo")

def perguntar(q, k=5):
    o = answer(mascarar_pii(q.strip()), gen, retriever, mode="rag", threshold=TH, k=k)
    print("\n" + (o["resposta"] or "(não consegui gerar uma resposta válida; tente reformular)"))
    for c in o.get("citacoes", []):
        ini, fim = c["paginas"]
        print(f"   fonte: {c['doc_id']} · {c['titulo']}, " + (f"p. {ini}" if ini == fim else f"pp. {ini}-{fim}"))
    for a in o.get("avisos", []):
        print("   aviso:", a)
    if o.get("top_score") is not None:
        print(f"   [confiança da busca: {o['top_score']:.3f}]")

print("\nEscreva sua pergunta (ou 'sair'). Dica: cite o documento, ex.: 'Segundo o SCI-300, ...'.")
while True:
    q = input("\nPergunta: ")
    if q.strip().lower() in ("sair", "exit", ""):
        break
    perguntar(q)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Pronto: 1312 trechos | limiar 0.836 | LoRA ativo

Escreva sua pergunta (ou 'sair'). Dica: cite o documento, ex.: 'Segundo o SCI-300, ...'.

O propósito de um Plano Operativo de Prevenção e Combate aos Incêndios Florestais é definir estratégias aplicáveis a cada período de estiagem que minimizem o risco de incêndios e seus impactos.
   fonte: DOC001 · Roteiro Metodológico para Elaboração de Planos Operativos, pp. 7-9
   [confiança da busca: 0.884]

De acordo com o SCI-200, a amplitude de controle recomendada para um supervisor no SCI varia de três a sete subordinados, sendo cinco o ideal.
   fonte: DOC007 · Sistema de Comando de Incidentes - Resposta Inicial, p. 110
   [confiança da busca: 0.925]
